# MSE 546 - Baseline Model: Detecting AI-Generated Text

**Team 13**

This notebook implements a simple baseline model for the Kaggle competition "LLM - Detect AI Generated Text".

**Approach:** TF-IDF Vectorization + Logistic Regression

---

## Step 0: Setup for Google Colab

**If running in Google Colab**, run the cell below to upload your data file.

**If running locally** (Jupyter on your computer), skip this cell.

In [ ]:
# ============================================================
# GOOGLE COLAB SETUP - Run this cell if using Colab
# ============================================================

import os

# Check if we're running in Google Colab
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab")
except ImportError:
    IN_COLAB = False
    print("Running locally (not in Colab)")

if IN_COLAB:
    from google.colab import files
    
    # Check if data file already exists
    if not os.path.exists('train_v4_drcat_01.csv'):
        print("\n" + "="*50)
        print("Please upload 'train_v4_drcat_01.csv'")
        print("(from your DAIGT-V4-TRAIN-DATASEt folder)")
        print("="*50 + "\n")
        uploaded = files.upload()
        print("\nFile uploaded successfully!")
    else:
        print("Data file already exists, skipping upload.")

---

## Step 1: Import Libraries and Load Data

We start by importing the necessary libraries:
- `pandas`: For data manipulation
- `sklearn`: For machine learning (TF-IDF, Logistic Regression, metrics)
- `matplotlib/seaborn`: For visualization

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn imports
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve
)

# Set random seed for reproducibility
RANDOM_STATE = 42

print("Libraries imported successfully!")

### Load the External Dataset

We use the DAIGT-V4 dataset which contains:
- ~27,000 human-written essays (from PERSUADE corpus)
- ~46,000 AI-generated essays (from 20+ different LLMs)

This balanced dataset allows us to train a meaningful classifier.

In [ ]:
# ============================================================
# Load Data - Automatically detects Colab vs Local
# ============================================================

import os

# Try different paths (Colab vs Local)
possible_paths = [
    "train_v4_drcat_01.csv",                              # Colab (uploaded file)
    "DAIGT-V4-TRAIN-DATASEt/train_v4_drcat_01.csv",       # Local (subfolder)
    "../DAIGT-V4-TRAIN-DATASEt/train_v4_drcat_01.csv",    # Local (parent folder)
]

data_path = None
for path in possible_paths:
    if os.path.exists(path):
        data_path = path
        break

if data_path is None:
    raise FileNotFoundError(
        "Could not find data file! Please either:\n"
        "1. Upload 'train_v4_drcat_01.csv' in Colab, or\n"
        "2. Make sure the DAIGT-V4 folder is in the correct location locally"
    )

print(f"Loading data from: {data_path}")
df = pd.read_csv(data_path)

# Display basic info
print(f"\nDataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Check class distribution
print("Class Distribution:")
print(df['label'].value_counts())
print(f"\nPercentages:")
print(df['label'].value_counts(normalize=True).mul(100).round(2))

### Prepare the Data

We only need two columns:
- `text`: The essay content (our input/features)
- `label`: 0 = human-written, 1 = AI-generated (our target)

In [ ]:
# Keep only the columns we need
X = df['text']  # Features (the essay text)
y = df['label']  # Target (0 = human, 1 = AI)

print(f"Number of essays: {len(X)}")
print(f"Number of labels: {len(y)}")

# Check for any missing values
print(f"\nMissing values in text: {X.isna().sum()}")
print(f"Missing values in label: {y.isna().sum()}")

---

## Step 2: Train-Test Split

We split our data into:
- **Training set (80%)**: Used to train the model
- **Test set (20%)**: Used to evaluate the model (never seen during training)

We use **stratified sampling** to ensure both sets have the same proportion of human vs AI essays.

In [ ]:
# Split data: 80% train, 20% test
# stratify=y ensures both sets have same class proportions
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.2,           # 20% for testing
    random_state=RANDOM_STATE,  # For reproducibility
    stratify=y               # Maintain class proportions
)

print(f"Training set size: {len(X_train)} essays")
print(f"Test set size: {len(X_test)} essays")

# Verify stratification worked
print(f"\nTraining set class distribution:")
print(y_train.value_counts(normalize=True).mul(100).round(2))
print(f"\nTest set class distribution:")
print(y_test.value_counts(normalize=True).mul(100).round(2))

---

## Step 3: TF-IDF Vectorization

**TF-IDF** (Term Frequency - Inverse Document Frequency) converts text into numerical features.

- **TF**: How often a word appears in one document
- **IDF**: How rare a word is across all documents
- **TF-IDF = TF x IDF**: Words that are frequent in one doc but rare overall get high scores

**Parameters we use:**
- `max_features=10000`: Keep only the 10,000 most important words (reduces dimensionality)
- `ngram_range=(1,2)`: Include single words AND two-word phrases (bigrams)
- `min_df=3`: Ignore words that appear in fewer than 3 documents (removes typos/rare words)
- `max_df=0.9`: Ignore words that appear in >90% of documents (removes common words like "the")

In [ ]:
# Create TF-IDF vectorizer
tfidf = TfidfVectorizer(
    max_features=10000,    # Limit vocabulary size
    ngram_range=(1, 2),    # Use unigrams and bigrams
    min_df=3,              # Ignore rare words
    max_df=0.9,            # Ignore very common words
    stop_words='english'   # Remove common English words (the, is, at, etc.)
)

# Fit on training data and transform
# IMPORTANT: We fit ONLY on training data to avoid data leakage
print("Fitting TF-IDF on training data...")
X_train_tfidf = tfidf.fit_transform(X_train)

# Transform test data (using vocabulary learned from training)
print("Transforming test data...")
X_test_tfidf = tfidf.transform(X_test)

print(f"\nTraining data shape: {X_train_tfidf.shape}")
print(f"Test data shape: {X_test_tfidf.shape}")
print(f"\nEach essay is now represented as a vector of {X_train_tfidf.shape[1]} features")

In [ ]:
# Let's look at some of the features (words/phrases) learned
feature_names = tfidf.get_feature_names_out()
print(f"Total vocabulary size: {len(feature_names)}")
print(f"\nSample features (words/phrases):")
print(list(feature_names[:20]))  # First 20
print("...")
print(list(feature_names[-20:]))  # Last 20

---

## Step 4: Train Logistic Regression Model

**Logistic Regression** learns a weighted combination of features to predict probability of AI-generated text.

```
P(AI) = sigmoid(w1 x word1 + w2 x word2 + ... + bias)
```

Words with positive weights indicate AI text; words with negative weights indicate human text.

In [ ]:
# Create and train logistic regression model
model = LogisticRegression(
    max_iter=1000,           # Maximum iterations for convergence
    random_state=RANDOM_STATE,
    C=1.0,                   # Regularization strength (default)
    solver='lbfgs'           # Optimization algorithm
)

print("Training Logistic Regression model...")
model.fit(X_train_tfidf, y_train)
print("Training complete!")

### Inspect Model Weights

Let's see which words the model considers most indicative of AI vs human writing.

In [ ]:
# Get feature weights
weights = model.coef_[0]
feature_weights = list(zip(feature_names, weights))

# Sort by weight
feature_weights_sorted = sorted(feature_weights, key=lambda x: x[1])

# Words most indicative of HUMAN writing (most negative weights)
print("Top 15 words indicating HUMAN writing:")
print("-" * 40)
for word, weight in feature_weights_sorted[:15]:
    print(f"{word:25} {weight:.4f}")

print("\n")

# Words most indicative of AI writing (most positive weights)
print("Top 15 words indicating AI writing:")
print("-" * 40)
for word, weight in feature_weights_sorted[-15:]:
    print(f"{word:25} {weight:.4f}")

---

## Step 5: Evaluate Model Performance

We evaluate on the **test set** (data the model has never seen) using:
- **ROC-AUC**: How well does the model rank AI essays above human essays?
- **Precision**: Of essays predicted as AI, what % actually were AI?
- **Recall**: Of all actual AI essays, what % did we catch?
- **F1-Score**: Harmonic mean of precision and recall

In [ ]:
# Make predictions on test set
y_pred = model.predict(X_test_tfidf)           # Binary predictions (0 or 1)
y_pred_proba = model.predict_proba(X_test_tfidf)[:, 1]  # Probability of class 1 (AI)

# Calculate metrics
roc_auc = roc_auc_score(y_test, y_pred_proba)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

# Display results
print("=" * 50)
print("BASELINE MODEL PERFORMANCE")
print("=" * 50)
print(f"\nROC-AUC Score:  {roc_auc:.4f}")
print(f"Precision:      {precision:.4f}")
print(f"Recall:         {recall:.4f}")
print(f"F1-Score:       {f1:.4f}")
print("=" * 50)

In [ ]:
# Detailed classification report
print("\nDetailed Classification Report:")
print("-" * 50)
print(classification_report(y_test, y_pred, target_names=['Human (0)', 'AI (1)']))

### Confusion Matrix

The confusion matrix shows:
- **True Negatives (TN)**: Human essays correctly identified as human
- **False Positives (FP)**: Human essays incorrectly flagged as AI
- **False Negatives (FN)**: AI essays missed (incorrectly labeled as human)
- **True Positives (TP)**: AI essays correctly identified as AI

In [ ]:
# Create confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Human (0)', 'AI (1)'],
            yticklabels=['Human (0)', 'AI (1)'],
            annot_kws={'size': 14})
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.title('Confusion Matrix - Baseline Model', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Print raw numbers
tn, fp, fn, tp = cm.ravel()
print(f"\nTrue Negatives (Human->Human):  {tn}")
print(f"False Positives (Human->AI):    {fp}")
print(f"False Negatives (AI->Human):    {fn}")
print(f"True Positives (AI->AI):        {tp}")

### ROC Curve

The ROC curve shows the tradeoff between True Positive Rate (catching AI essays) and False Positive Rate (falsely flagging human essays) at different classification thresholds.

In [ ]:
# Calculate ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)

# Plot ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC Curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', label='Random Classifier')
plt.fill_between(fpr, tpr, alpha=0.3)
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve - Baseline Model', fontsize=14, fontweight='bold')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## Step 6: Summary for Proposal

Copy this section into your P2 proposal document.

In [ ]:
# Generate summary for proposal
print("=" * 60)
print("SUMMARY FOR P2 PROPOSAL")
print("=" * 60)
print(f"""
Simple Baseline: TF-IDF + Logistic Regression

Dataset:
- Total essays: {len(df):,}
- Human-written: {(y == 0).sum():,} ({(y == 0).mean()*100:.1f}%)
- AI-generated: {(y == 1).sum():,} ({(y == 1).mean()*100:.1f}%)
- Train/Test split: 80/20 (stratified)

Model Configuration:
- Vectorization: TF-IDF (max 10,000 features, unigrams + bigrams)
- Classifier: Logistic Regression

Performance Metrics:
- ROC-AUC:   {roc_auc:.4f}
- Precision: {precision:.4f}
- Recall:    {recall:.4f}
- F1-Score:  {f1:.4f}

Confusion Matrix:
- True Negatives:  {tn:,}
- False Positives: {fp:,}
- False Negatives: {fn:,}
- True Positives:  {tp:,}
""")
print("=" * 60)